In [1]:
!pip install optuna lightgbm pandas scikit-learn matplotlib

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load data (replace with your dataset path)
X_train = pd.read_parquet('temp/X_train.parquet')
X_val = pd.read_parquet('temp/X_val.parquet')


y_train = X_train['TARGET']
y_val = X_val['TARGET']

X_train = X_train.drop(columns=['TARGET'])
X_val = X_val.drop(columns=['TARGET'])

In [3]:
y_train.isnull().sum(), y_val.isnull().sum()

(0, 0)

In [4]:
X_train.columns = X_train.columns.str.replace(r'[^\w]', '_', regex=True)
X_val.columns = X_val.columns.str.replace(r'[^\w]', '_', regex=True)

In [5]:
import optuna
import lightgbm as lgb
import numpy as np

def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 2e-2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 10, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.8),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 200),
        'n_jobs': -1
    }
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        param,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=200,
    )
    
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    return auc


In [6]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Print the best parameters
print("Best parameters:", study.best_params)
print("Best AUC:", study.best_value)

[I 2024-12-06 20:00:18,346] A new study created in memory with name: no-name-aee7d597-4cd0-4857-b326-d2e3f9ca7199


[LightGBM] [Warning] min_data_in_leaf is set=160, min_child_samples=108 will be ignored. Current value: min_data_in_leaf=160
[LightGBM] [Warning] min_data_in_leaf is set=160, min_child_samples=108 will be ignored. Current value: min_data_in_leaf=160
[LightGBM] [Info] Number of positive: 226133, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.872098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 155348
[LightGBM] [Info] Number of data points in the train set: 452266, number of used features: 994
[LightGBM] [Warning] min_data_in_leaf is set=160, min_child_samples=108 will be ignored. Current value: min_data_in_leaf=160
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


[I 2024-12-06 20:01:04,555] Trial 0 finished with value: 0.7551618198371788 and parameters: {'learning_rate': 0.0019955524754715352, 'num_leaves': 66, 'max_depth': 11, 'min_data_in_leaf': 160, 'feature_fraction': 0.5392551136075034, 'bagging_fraction': 0.7051629378818959, 'bagging_freq': 3, 'lambda_l1': 0.016311454664784265, 'lambda_l2': 1.0980750498198717, 'min_child_samples': 108}. Best is trial 0 with value: 0.7551618198371788.


[LightGBM] [Warning] min_data_in_leaf is set=144, min_child_samples=140 will be ignored. Current value: min_data_in_leaf=144
[LightGBM] [Warning] min_data_in_leaf is set=144, min_child_samples=140 will be ignored. Current value: min_data_in_leaf=144
[LightGBM] [Info] Number of positive: 226133, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.674830 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 155348
[LightGBM] [Info] Number of data points in the train set: 452266, number of used features: 994
[LightGBM] [Warning] min_data_in_leaf is set=144, min_child_samples=140 will be ignored. Current value: min_data_in_leaf=144
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


[I 2024-12-06 20:02:07,208] Trial 1 finished with value: 0.7712379638535902 and parameters: {'learning_rate': 0.011421490959932352, 'num_leaves': 164, 'max_depth': 15, 'min_data_in_leaf': 144, 'feature_fraction': 0.5685711291935323, 'bagging_fraction': 0.7090017392435879, 'bagging_freq': 7, 'lambda_l1': 0.03197214958672289, 'lambda_l2': 0.23824433109787707, 'min_child_samples': 140}. Best is trial 1 with value: 0.7712379638535902.


...

[LightGBM] [Warning] min_data_in_leaf is set=138, min_child_samples=165 will be ignored. Current value: min_data_in_leaf=138
[LightGBM] [Warning] min_data_in_leaf is set=138, min_child_samples=165 will be ignored. Current value: min_data_in_leaf=138
[LightGBM] [Info] Number of positive: 226133, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.795485 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 155348
[LightGBM] [Info] Number of data points in the train set: 452266, number of used features: 994
[LightGBM] [Warning] min_data_in_leaf is set=138, min_child_samples=165 will be ignored. Current value: min_data_in_leaf=138
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


[I 2024-12-06 22:06:02,898] Trial 98 finished with value: 0.7625046494347377 and parameters: {'learning_rate': 0.0029366066415559187, 'num_leaves': 187, 'max_depth': 15, 'min_data_in_leaf': 138, 'feature_fraction': 0.5510986141027662, 'bagging_fraction': 0.8596453696354436, 'bagging_freq': 1, 'lambda_l1': 0.9703430360985306, 'lambda_l2': 0.9073529714592422, 'min_child_samples': 165}. Best is trial 86 with value: 0.7789777088565315.


[LightGBM] [Warning] min_data_in_leaf is set=135, min_child_samples=160 will be ignored. Current value: min_data_in_leaf=135
[LightGBM] [Warning] min_data_in_leaf is set=135, min_child_samples=160 will be ignored. Current value: min_data_in_leaf=135
[LightGBM] [Info] Number of positive: 226133, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.873216 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 155348
[LightGBM] [Info] Number of data points in the train set: 452266, number of used features: 994
[LightGBM] [Warning] min_data_in_leaf is set=135, min_child_samples=160 will be ignored. Current value: min_data_in_leaf=135
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


[I 2024-12-06 22:07:12,829] Trial 99 finished with value: 0.7771471795544445 and parameters: {'learning_rate': 0.017515628947563616, 'num_leaves': 196, 'max_depth': 15, 'min_data_in_leaf': 135, 'feature_fraction': 0.5713317503199303, 'bagging_fraction': 0.8306624557807177, 'bagging_freq': 1, 'lambda_l1': 0.7113703736778064, 'lambda_l2': 1.308105905608443, 'min_child_samples': 160}. Best is trial 86 with value: 0.7789777088565315.


Best parameters: {'learning_rate': 0.019888638222992847, 'num_leaves': 191, 'max_depth': 15, 'min_data_in_leaf': 119, 'feature_fraction': 0.5927892648547602, 'bagging_fraction': 0.9969565014373664, 'bagging_freq': 6, 'lambda_l1': 0.697976393699385, 'lambda_l2': 2.6587476507796275, 'min_child_samples': 159}
Best AUC: 0.7789777088565315


In [7]:
best_params = study.best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

In [8]:
best_params

{'learning_rate': 0.019888638222992847,
 'num_leaves': 191,
 'max_depth': 15,
 'min_data_in_leaf': 119,
 'feature_fraction': 0.5927892648547602,
 'bagging_fraction': 0.9969565014373664,
 'bagging_freq': 6,
 'lambda_l1': 0.697976393699385,
 'lambda_l2': 2.6587476507796275,
 'min_child_samples': 159,
 'objective': 'binary',
 'metric': 'auc'}

In [9]:
best_params = best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

print("Training the final model with the best parameters")
print(best_params)

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=300
)

Training the final model with the best parameters
{'learning_rate': 0.019888638222992847, 'num_leaves': 191, 'max_depth': 15, 'min_data_in_leaf': 119, 'feature_fraction': 0.5927892648547602, 'bagging_fraction': 0.9969565014373664, 'bagging_freq': 6, 'lambda_l1': 0.697976393699385, 'lambda_l2': 2.6587476507796275, 'min_child_samples': 159, 'objective': 'binary', 'metric': 'auc'}
[LightGBM] [Warning] min_data_in_leaf is set=119, min_child_samples=159 will be ignored. Current value: min_data_in_leaf=119
[LightGBM] [Warning] min_data_in_leaf is set=119, min_child_samples=159 will be ignored. Current value: min_data_in_leaf=119
[LightGBM] [Info] Number of positive: 226133, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.782954 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 155348
[LightGBM] [Info] Number of data points in the train set: 452266, number of used features: 994
[Li

In [10]:
y_pred = final_model.predict(X_val)

In [11]:
roc_auc_score(y_val,y_pred)

0.7837751512476935

In [12]:
# Get feature importance and feature names
importance = final_model.feature_importance(importance_type='gain')  # 'gain' measures the contribution
importance_split = final_model.feature_importance(importance_type='split')
feature_names = X_train.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance,
    'Num_Split': importance_split
}).sort_values(by='Importance', ascending=False)

# Display the most important feature
best_feature = importance_df.iloc[0]
print(f"The most important feature is: {best_feature['Feature']} with an importance score of {best_feature['Importance']}")

# Optional: Display the top 5 features
print("\nTop 5 Features:")
print(importance_df.head())


The most important feature is: WEAK_FEATURE with an importance score of 1070474.4112358093

Top 5 Features:
              Feature    Importance  Num_Split
1011     WEAK_FEATURE  1.070474e+06        632
866   EXT_SOURCE_MEAN  5.276105e+05        370
1012   WEAK_FEATURE_2  1.911226e+05        782
864    EXT_SOURCE_SUM  1.316560e+05        409
865   EXT_SOURCE_PROD  8.040409e+04        307


In [13]:
importance_df.to_excel('temp/feature_importance_big.xlsx', index=False)